In [ ]:
# Cell 1: Installs and Imports
!pip install --upgrade pip # Ensure pip is up-to-date
!pip install -U --force-reinstall openai-whisper TTS webrtcvad # Force reinstall these
!pip install -U --force-reinstall ffmpeg-python # Force reinstall this

# The force-reinstall flag will ensure that dependencies like numpy and numba are also re-evaluated and potentially reinstalled

In [ ]:
!pip install -U --force-reinstall together googletrans==4.0.0rc1 requests # Force reinstall these

In [ ]:
!pip uninstall -y numpy # Force uninstall any existing numpy versions
!pip install -U numpy # Install the latest numpy first
!pip install --upgrade pip # Ensure pip is up-to-date

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
import os
import numpy as np
from IPython.display import HTML, Audio, display
from google.colab.output import eval_js
from base64 import b64decode
import json # For the agent
import requests # For the agent
from googletrans import Translator # For the agent
from together import Together # For the agent

# Standard Whisper and TTS imports
import whisper
from TTS.api import TTS

print("All necessary libraries installed/imported.")

All necessary libraries installed/imported.


In [2]:
# Cell 2: Colab Audio Recording Function (Corrected Version)
# Using the corrected version from our previous discussion

AUDIO_HTML_RECORDER_CORRECTED = """
<script>
// Global variables to manage recorder state
var colab_recorder_instance = null;
var colab_stream_instance = null;
var colab_audio_chunks = [];

// These will hold the resolve/reject functions for the current Promise
var colab_resolve_audio_promise = null;
var colab_reject_audio_promise = null;

// This master function is called by Python's eval_js.
function colab_js_recordAudio() {
  return new Promise((resolve, reject) => {
    colab_resolve_audio_promise = resolve;
    colab_reject_audio_promise = reject;

    colab_audio_chunks = [];
    const statusElement = document.getElementById("colabAudioRecordingStatus");
    const stopButton = document.getElementById("colabStopAudioRecordingButton");

    if (statusElement) statusElement.innerText = "Initializing microphone...";
    if (stopButton) stopButton.disabled = true;

    var constraints = { audio: true, video: false };
    navigator.mediaDevices.getUserMedia(constraints)
      .then(function(stream) {
        colab_stream_instance = stream;
        colab_recorder_instance = new MediaRecorder(stream, { mimeType: 'audio/webm' });

        colab_recorder_instance.ondataavailable = function(e) {
          if (e.data.size > 0) { colab_audio_chunks.push(e.data); }
        };

        colab_recorder_instance.onstop = function(e) {
          var blob = new Blob(colab_audio_chunks, { 'type': 'audio/webm' });
          colab_audio_chunks = [];
          var reader = new FileReader();
          reader.onloadend = function() {
            var base64Audio = reader.result.split(',')[1];
            if (colab_resolve_audio_promise) { colab_resolve_audio_promise(base64Audio); }
          };
          reader.onerror = function(error) {
            if (colab_reject_audio_promise) { colab_reject_audio_promise("FileReader error: " + error.toString()); }
          };
          reader.readAsDataURL(blob);
          if (statusElement) statusElement.innerText = "Audio captured, processing...";
        };

        colab_recorder_instance.onerror = function(event) {
            if (colab_reject_audio_promise) { colab_reject_audio_promise('MediaRecorder error: ' + event.error.name + " - " + event.error.message); }
            if (statusElement) statusElement.innerText = "Recorder error: " + event.error.name;
        };

        colab_recorder_instance.start();
        if (statusElement) statusElement.innerText = "Recording... Click 'Stop Recording' below.";
        if (stopButton) stopButton.disabled = false;
      })
      .catch(function(err) {
        if (statusElement) statusElement.innerText = "Microphone access error: " + err.message;
        if (stopButton) stopButton.disabled = true;
        if (colab_reject_audio_promise) { colab_reject_audio_promise("Microphone access error: " + err.message); }
      });
  });
}

function colab_js_stopRecording() {
  const statusElement = document.getElementById("colabAudioRecordingStatus");
  const stopButton = document.getElementById("colabStopAudioRecordingButton");
  if (colab_recorder_instance && colab_recorder_instance.state === "recording") {
    colab_recorder_instance.stop();
  }
  if (colab_stream_instance) {
    colab_stream_instance.getTracks().forEach(track => track.stop());
  }
  if (statusElement) statusElement.innerText = "Recording stopped by user.";
  if (stopButton) stopButton.disabled = true;
}
</script>
<div>
  <div id="colabAudioRecordingStatus" style="margin-bottom:10px;">Python will initiate recording.</div>
  <button id="colabStopAudioRecordingButton" onclick="colab_js_stopRecording()" disabled style="padding:8px 12px; background-color:#f44336; color:white; border:none; border-radius:4px; cursor:pointer;">Stop Recording</button>
</div>
"""

def record_colab_audio(filename="colab_recorded_audio.webm"):
  display(HTML(AUDIO_HTML_RECORDER_CORRECTED))
  print("Please grant microphone access in your browser if prompted.")
  print("Recording will start automatically. Speak, then click 'Stop Recording' in the UI above.")
  try:
    base64_audio_data = eval_js('colab_js_recordAudio()')
    audio_bytes = b64decode(base64_audio_data)
    with open(filename, 'wb') as f: f.write(audio_bytes)
    print(f"Audio recorded and saved as '{filename}' ({len(audio_bytes)} bytes)")
    return filename
  except Exception as e:
    print(f"Error during audio recording: {e}")
    try:
        eval_js("""
            if (typeof colab_stream_instance !== 'undefined' && colab_stream_instance) {
                colab_stream_instance.getTracks().forEach(track => track.stop());}
            colab_resolve_audio_promise = null; colab_reject_audio_promise = null;
            const stopBtn = document.getElementById("colabStopAudioRecordingButton"); if(stopBtn) stopBtn.disabled = true;
            const statusElm = document.getElementById("colabAudioRecordingStatus"); if(statusElm) statusElm.innerText = "Ready or error.";
        """)
    except Exception as cleanup_e: print(f"Error during JS cleanup: {cleanup_e}")
    return None

print("Colab audio recording function defined.")

Colab audio recording function defined.


In [18]:
import json
# requests should be imported by standalone functions or at the top of the cell
from googletrans import Translator
from together import Together
# datetime is imported by standalone functions or at the top of the cell

class CustomSmartHomeAgent:
    def __init__(self, llm_api_key="6ba1bc9d36c18b34a8e1a820d132b0b3614192ca291165e56799eff76f03304b", primary_llm_language='en'): # Replace with your actual key
        self.llm_api_key = llm_api_key
        self.primary_llm_language = primary_llm_language
        try:
            self.llm_client = Together(api_key=self.llm_api_key)
            print(f"[AGENT_DEBUG] CustomSmartHomeAgent initialized. LLM lang: {self.primary_llm_language}")
        except Exception as e:
            print(f"[AGENT_DEBUG] ERROR Initializing Together client: {e}")
            self.llm_client = None # Set client to None on failure

        self.valid_devices = {
            "lamp": ["kitchen", "bathroom", "room 1", "room 2" , "room one" , "room two"],
            "ac": ["room 1", "kitchen"],
            "tv": ["living room"]
        }

        self.device_states = {}
        for device_type, locations in self.valid_devices.items():
            for loc_name in locations:
              if loc_name == "room one" or loc_name== "room two":
                continue
              else:
                device_key = f"{loc_name.lower().replace(' ', '_')}_{device_type}"
                state_detail = {"status": "off"}
                if device_type == "ac":
                    state_detail["temperature"] = 20
                if device_type == "lamp" and loc_name.lower() in ["bathroom", "room 2"]:
                     state_detail["brightness"] = 75
                self.device_states[device_key] = state_detail
        print(f"[AGENT_DEBUG] Initial device states: {self.device_states}")
        self.original_lan = None

    def _is_valid_device_location(self, device, location):
        if not device or not location:
            return False
        normalized_location = location.lower()
        is_valid = normalized_location in [loc.lower() for loc in self.valid_devices.get(device, [])]
        print(f"[AGENT_DEBUG] _is_valid_device_location: device='{device}', location='{location}', normalized='{normalized_location}', valid={is_valid}")
        return is_valid

    def _detect_language(self, text):
        if any('\u0600' <= char <= '\u06FF' for char in text):
            self.original_lan = 'fa'
            return 'fa'
        self.original_lan = 'en'
        return 'en'

    def _translate_text(self, text, source_lang, target_lang):
        if source_lang == target_lang:
            return text
        try:
            translator = Translator()
            result = translator.translate(text, src=source_lang, dest=target_lang)
            print(f"[AGENT_DEBUG] Translated '{text}' from {source_lang} to {target_lang}: '{result.text}'")
            return result.text
        except Exception as e:
            print(f"[AGENT_DEBUG] ERROR _translate_text: from {source_lang} to {target_lang} for '{text}': {e}")
            # Return a recognizable error string if translation fails
            return f"TranslationError_{text}_to_{target_lang}"

    def _get_llm_analysis(self, command_text_for_llm):
        if not self.llm_client:
             print(f"[AGENT_DEBUG] _get_llm_analysis: LLM client not initialized.")
             return {"intent": "error", "message": "LLM client not available."}

        prompt = f"""
        You are an AI assistant for a smart home. Analyze the user's command.
        Respond ONLY with a JSON object containing:
        - "intent": (e.g., "control_device", "get_weather", "get_news", "get_time", "unknown")
        - "device": (e.g., "lamp", "ac", "tv", "none") - type of device.
        - "action": (e.g., "turn_on", "turn_off", "set_temperature", "set_brightness", "fetch", "none") - action to perform.
        - "parameters": A dictionary of parameters. Examples:
            {{"location": "kitchen"}}
            {{"location": "room 1", "temperature": 22}} (for AC temperature)
            {{"location": "room one", "temperature": 22}} (for AC temperature)
            {{"location": "bathroom", "brightness": 60}} (for lamp brightness)
            {{"topic": "technology"}} (for news)
            {{"location": "london"}} (for weather)
        If a parameter is not present in the command, do not include it in the parameters dictionary.
        Limit your JSON explanation to 10 words or less in each value.

        User command: "{command_text_for_llm}"
        JSON Analysis:
        """
        print(f"[AGENT_DEBUG] _get_llm_analysis: Sending to LLM. Command: \"{command_text_for_llm}\"")
        try:
            response = self.llm_client.chat.completions.create(
                model="meta-llama/Llama-3-70b-chat-hf",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.4,
                response_format={"type": "json_object"}
            )
            llm_output_str = response.choices[0].message.content
            print(f"[AGENT_DEBUG] _get_llm_analysis: LLM Raw Output: {llm_output_str}")
            parsed_output = json.loads(llm_output_str)
            print(f"[AGENT_DEBUG] _get_llm_analysis: LLM Parsed Output: {parsed_output}")
            return parsed_output
        except json.JSONDecodeError as je:
             print(f"[AGENT_DEBUG] ERROR _get_llm_analysis: JSON decoding error: {je}. Raw output: {llm_output_str}")
             return {"intent": "error", "message": f"LLM returned invalid JSON: {str(je)}"}
        except Exception as e:
            print(f"[AGENT_DEBUG] ERROR _get_llm_analysis: LLM call error: {e}")
            return {"intent": "error", "message": f"LLM call error: {str(e)}"}


    def execute_command(self, natural_language_command):
        print(f"[AGENT_DEBUG] execute_command: Received original command: '{natural_language_command}'")
        original_lang = self._detect_language(natural_language_command)
        print(f"[AGENT_DEBUG] execute_command: Detected lang: {original_lang}")

        # Translate to primary LLM language if necessary
        command_for_llm = natural_language_command
        if original_lang != self.primary_llm_language:
            command_for_llm = self._translate_text(natural_language_command, original_lang, self.primary_llm_language)

        if not command_for_llm or "TranslationError" in command_for_llm :
            print(f"[AGENT_DEBUG] execute_command: Error in command after translation: '{command_for_llm}'")
            response_text = self._generate_response("translation_error", original_lang, {})
            print(f"[AGENT_DEBUG] execute_command: Returning due to translation/empty command. Response: '{response_text}', States: {self.device_states}")
            return response_text, self.device_states

        analysis = self._get_llm_analysis(command_for_llm)

        # Handle LLM analysis errors early
        if analysis.get("intent") == "error":
            print(f"[AGENT_DEBUG] execute_command: LLM analysis returned an error: {analysis.get('message', 'Unknown LLM error')}")
            response_text = self._generate_response("error", original_lang, {"message": analysis.get("message", "Unknown LLM error")})
            print(f"[AGENT_DEBUG] execute_command: Returning due to LLM error. Response: '{response_text}', States: {self.device_states}")
            return response_text, self.device_states

        intent = analysis.get("intent")
        device_type = analysis.get("device")
        action = analysis.get("action")
        parameters = analysis.get("parameters", {})

        execution_result_message = "Action status pending."

        if intent == "control_device":
            location_from_params = parameters.get("location", "")
            # Ensure location is in parameters for _is_valid_device_location check
            if not location_from_params and "location" in parameters: # If location was None or empty string
                 location_from_params = "" # Treat as invalid location
            elif not location_from_params and "location" not in parameters:
                 location_from_params = "" # No location provided by LLM

            if not self._is_valid_device_location(device_type, location_from_params):
                print(f"[AGENT_DEBUG] execute_command: Invalid device/location for control: {device_type} in {location_from_params}")
                final_response_text = self._generate_response("unknown_device", original_lang, {"device_name": f"{device_type} in {location_from_params}"})
                # No state change on unknown device, return current states
                return final_response_text, self.device_states

            # Execute control based on device type and update states
            if device_type == "lamp":
                execution_result_message = self._control_lamp(action, parameters)
            elif device_type == "ac":
                execution_result_message = self._control_ac(action, parameters)
            elif device_type == "tv":
                execution_result_message = self._control_tv(action, parameters)
            else:
                # Should ideally be caught by _is_valid_device_location, but as a fallback
                intent = "unknown_device"
                parameters["device_name"] = device_type
                execution_result_message = "Device type recognized by LLM but not specifically handled for control."
            print(f"[AGENT_DEBUG] execute_command: Control device execution message: '{execution_result_message}'")

        elif intent == "get_weather":
            execution_result_message = self._get_weather(parameters)
            print(f"[AGENT_DEBUG] execute_command: Get weather execution message: '{execution_result_message}'")
        elif intent == "get_news":
            execution_result_message = self._get_news(parameters)
            print(f"[AGENT_DEBUG] execute_command: Get news execution message: '{execution_result_message}'")
        elif intent == "get_time":
            execution_result_message = self._get_time(parameters)
            print(f"[AGENT_DEBUG] execute_command: Get time execution message: '{execution_result_message}'")
        else: # unknown intent
            execution_result_message = f"Intent '{intent}' not understood."
            print(f"[AGENT_DEBUG] execute_command: Unknown intent: '{execution_result_message}'")


        # Generate the final response text based on the original language
        final_response_text = self._generate_response(intent if intent else "unknown", original_lang, parameters, execution_result_message, device_type, action)
        print(f"[AGENT_DEBUG] execute_command: Final response text: '{final_response_text}'")
        print(f"[AGENT_DEBUG] execute_command: Returning final states: {self.device_states}")

        # Always return the latest device states, even if the command was unknown or an error occurred
        # return final_response_text, self.device_states
        # final_state = self.device_states
        final_state = self.device_states
        print(f"[AGENT_DEBUG] execute_command: Returning final states: {final_state}")




        return final_response_text

    def status(self):
      dictionary_as_string = json.dumps(self.device_states)
      print(f"[AGENT_DEBUG] status:Coming")
      print(f"[AGENT_DEBUG] status: Returning dictionary_as_string: {dictionary_as_string}")
      return self.device_states

    def _generate_response(self, intent, lang, params, execution_message="N/A", device_name=None, action_name=None):
        # This method is mostly the same as your last version, ensure it returns a string.
        # Adding a print here to see what it's about to return.
        response_text = ""
        location = params.get("location", "مکان نامشخص" if lang == 'fa' else "unspecified location")
        topic = params.get("topic")
        brightness = params.get("brightness")
        temperature = params.get("temperature")

        device_fa_map = {"lamp": "چراغ", "ac": "کولر", "tv": "تلویزیون"}
        action_fa_map = {"turn_on": "روشن کردن", "turn_off": "خاموش کردن",
                         "set_temperature": "تنظیم دمای", "set_brightness": "تنظیم روشنایی"}
        device_name_fa = device_fa_map.get(device_name, device_name if device_name else ("دستگاه" if lang == 'fa' else "device"))
        action_name_fa = action_fa_map.get(action_name, action_name if action_name else ("عملیات" if lang == 'fa' else "action"))


        # Ensure execution_message is translated if generating a Persian response and it's not already Persian
        translated_execution_message = execution_message
        if lang == 'fa' and execution_message and not any('\u0600' <= char <= '\u06FF' for char in execution_message) and "TranslationError" not in execution_message:
             # Simple heuristic: if it doesn't contain Persian chars and isn't a translation error message
             translated_execution_message = self._translate_text(execution_message, 'en', 'fa')
             if "TranslationError" in translated_execution_message: # If translation failed
                  translated_execution_message = "ترجمه نشد." # Indicate translation failed in Farsi


        if lang == 'en':
            if intent == "control_device":
                 # For control_device, the action name and device name are included in the template
                 # The execution_message contains the status like "turned on.", "brightness set to 75%."
                 response_text = f"Okay, the {device_name} in {location}. Status: {execution_message}"
            elif intent == "get_weather": response_text = f"{execution_message}" # Weather message is already formatted in _get_weather
            elif intent == "get_news": response_text = f"{execution_message}" # News message is already formatted in _get_news
            elif intent == "get_time": response_text = f"The current time is: {execution_message}"
            elif intent == "unknown_device": response_text = f"Sorry, I cannot control a device '{params.get('device_name', 'unknown device')}' at that location."
            elif intent == "translation_error": response_text = "I had trouble understanding after translation."
            elif intent == "error": response_text = f"An internal error occurred: {params.get('message', execution_message)}"
            else: response_text = "I'm sorry, I didn't understand that command."
        elif lang == 'fa':
             if intent == "control_device":
                 # Construct the Farsi response using the translated action/device names and the translated execution message
                 response_text = f"بسیار خب، وضعیت {device_name_fa} در {location}: {translated_execution_message}"
             elif intent == "get_weather": response_text = f"{translated_execution_message}"
             elif intent == "get_news": response_text = f"{translated_execution_message}"
             elif intent == "get_time":
                  # Translate time string - simple check if it looks like a time string
                  if ":" in execution_message:
                       response_text = f"ساعت فعلی: {self._translate_text(execution_message, 'en', 'fa')}"
                  else:
                       response_text = f"ساعت فعلی: {translated_execution_message}" # Use translated if not a typical time format
             elif intent == "unknown_device": response_text = f"متاسفم، امکان کنترل دستگاهی به نام '{params.get('device_name', 'ناشناس')}' در آن مکان وجود ندارد."
             elif intent == "translation_error": response_text = "پس از ترجمه، در درک دستور مشکل داشتم."
             elif intent == "error":
                  # Translate error message if it seems to be in English
                  error_msg_to_translate = params.get('message', execution_message)
                  if error_msg_to_translate and not any('\u0600' <= char <= '\u06FF' for char in error_msg_to_translate):
                       response_text = f"یک خطای داخلی رخ داد: {self._translate_text(error_msg_to_translate, 'en', 'fa')}"
                  else:
                       response_text = f"یک خطای داخلی رخ داد: {error_msg_to_translate}" # Use as is if already Persian or empty
             else: response_text = "متاسفم، متوجه دستور شما نشدم."


        final_response = response_text if response_text else ("Error generating response." if lang=='en' else "خطا در تولید پاسخ.")
        print(f"[AGENT_DEBUG] _generate_response: Intent='{intent}', Lang='{lang}', ExecMsg='{execution_message}', FinalResponse='{final_response}'")
        return final_response


    def _control_lamp(self, action, params):
        location_norm = params.get("location", "").lower().replace(' ', '_')
        if location_norm == "room_one": location_norm = "room_1"
        if location_norm == "room_two": location_norm = "room_2"
        device_key = f"{location_norm}_lamp"
        brightness_param = params.get("brightness")
        print(f"[AGENT_DEBUG] _control_lamp: Action='{action}', Location='{location_norm}', Brightness='{brightness_param}', DeviceKey='{device_key}'")

        if device_key in self.device_states:
            current_state = self.device_states[device_key]
            if action == "turn_on":
                current_state["status"] = "on"
                # If brightness is provided and the lamp type supports it, set it
                if brightness_param is not None and "brightness" in current_state:
                    try:
                        current_state["brightness"] = int(brightness_param)
                        return f"turned on and brightness set to {current_state['brightness']}%."
                    except ValueError:
                        print(f"[AGENT_DEBUG] _control_lamp: Invalid brightness value: {brightness_param}")
                        return f"turned on, but invalid brightness value '{brightness_param}' ignored."
                return "turned on."
            elif action == "turn_off":
                current_state["status"] = "off"
                return "turned off."
            elif action == "set_brightness" and "brightness" in current_state:
                if brightness_param is not None:
                    try:
                        current_state["brightness"] = int(brightness_param)
                        # If setting brightness while off, assume intent is to turn on
                        if current_state["status"] == "off":
                            current_state["status"] = "on"
                            return f"brightness set to {current_state['brightness']}% and turned on."
                        return f"brightness set to {current_state['brightness']}%."
                    except ValueError:
                        print(f"[AGENT_DEBUG] _control_lamp: Invalid brightness value: {brightness_param}")
                        return f"invalid brightness value '{brightness_param}'. Brightness not changed."
                return "brightness level not specified for set_brightness action."
            return f"unknown action '{action}' for lamp."
        print(f"[AGENT_DEBUG] _control_lamp: Device key '{device_key}' not found in states.")
        return f"lamp at '{location_norm}' not found."


    def _control_ac(self, action, params):
        location_norm = params.get("location", "").lower().replace(' ', '_')
        device_key = f"{location_norm}_ac"
        temperature_param = params.get("temperature")
        print(f"[AGENT_DEBUG] _control_ac: Action='{action}', Location='{location_norm}', Temp='{temperature_param}', DeviceKey='{device_key}'")

        if device_key in self.device_states:
            current_state = self.device_states[device_key]
            if action == "turn_on":
                current_state["status"] = "on"
                if temperature_param is not None :
                     try:
                        current_state["temperature"] = int(temperature_param)
                        return f"turned on and temperature set to {current_state['temperature']}°C."
                     except ValueError:
                         print(f"[AGENT_DEBUG] _control_ac: Invalid temperature value: {temperature_param}")
                         return f"turned on, but invalid temperature value '{temperature_param}' ignored."
                return "turned on."
            elif action == "turn_off":
                current_state["status"] = "off"
                return "turned off."
            elif action == "set_temperature":
                if temperature_param is not None:
                    try:
                        current_state["temperature"] = int(temperature_param)
                        # If setting temperature while off, assume intent is to turn on
                        if current_state["status"] == "off":
                            current_state["status"] = "on"
                            return f"temperature set to {current_state['temperature']}°C and turned on."
                        return f"temperature set to {current_state['temperature']}°C."
                    except ValueError:
                         print(f"[AGENT_DEBUG] _control_ac: Invalid temperature value: {temperature_param}")
                         return f"invalid temperature value '{temperature_param}'. Temperature not changed."
                return "temperature not specified for set_temperature action."
            return f"unknown action '{action}' for AC."
        print(f"[AGENT_DEBUG] _control_ac: Device key '{device_key}' not found in states.")
        return f"AC at '{location_norm}' not found."

    def _control_tv(self, action, params):
        location_norm = params.get("location", "").lower().replace(' ', '_')
        device_key = f"{location_norm}_tv"
        print(f"[AGENT_DEBUG] _control_tv: Action='{action}', Location='{location_norm}', DeviceKey='{device_key}'")

        if device_key in self.device_states:
            current_state = self.device_states[device_key]
            if action == "turn_on":
                current_state["status"] = "on"
                return "turned on."
            elif action == "turn_off":
                current_state["status"] = "off"
                return "turned off."
            return f"unknown action '{action}' for TV."
        print(f"[AGENT_DEBUG] _control_tv: Device key '{device_key}' not found in states.")
        return f"TV at '{location_norm}' not found."

    def _get_weather(self, params):
        location = params.get("location")
        if not location: location = "Tehran"
        print(f"[AGENT_DEBUG] _get_weather (agent command): Fetching for '{location}'")
        # Use Open-Meteo geocoding to get latitude and longitude
        try:
            geo_resp = requests.get("https://geocoding-api.open-meteo.com/v1/search", params={"name": location, "count": 1, "language": "en", "format": "json"})
            geo_resp.raise_for_status()
            geo_data = geo_resp.json()
            if not geo_data.get("results"):
                print(f"[AGENT_DEBUG] _get_weather: Could not find location data for '{location}'")
                return f"Could not find location data for '{location}'."

            gr = geo_data["results"][0]
            lat, lon, dname = gr["latitude"], gr["longitude"], gr.get("name", location)

            weather_resp = requests.get("https://api.open-meteo.com/v1/forecast", params={"latitude": lat, "longitude": lon, "current_weather": True})
            weather_resp.raise_for_status()
            weather_data = weather_resp.json()

            if "current_weather" in weather_data:
                temp = weather_data["current_weather"]["temperature"]
                wind = weather_data["current_weather"]["windspeed"]
                weather_str = f"Current weather in {dname} is {temp}°C with wind {wind} km/h."
                print(f"[AGENT_DEBUG] _get_weather: Fetched weather: {weather_str}")
                return weather_str
            else:
                print(f"[AGENT_DEBUG] _get_weather: Could not retrieve current weather data for '{dname}'.")
                return f"Could not retrieve current weather for '{dname}'."
        except Exception as e:
            print(f"[AGENT_DEBUG] ERROR _get_weather: {e}")
            return f"Error fetching weather for {location}: {e}"


    def _get_news(self, params):
        topic = params.get("topic", "general")
        api_key = "9a332f8925de495f81a0ca08b526283b" # Use the configured API key
        url = "https://newsapi.org/v2/everything"
        print(f"[AGENT_DEBUG] _get_news (agent command): Fetching 1 news on '{topic}'")

        if not api_key or "YOUR_NEWS_API_KEY" in api_key:
            print("[AGENT_DEBUG] WARNING _get_news: NewsAPI key is a placeholder or missing.")
            return "NewsAPI key is not configured for agent commands."

        try:
            response = requests.get(url, params={"q": topic, "language": "en", "sortBy": "publishedAt", "pageSize": 1, "apiKey": api_key})
            response.raise_for_status()
            data = response.json()
            if data.get("status") == "ok" and data.get("articles"):
                article = data["articles"][0]
                title = article.get("title", "No title")
                description = article.get("description", "")
                print(f"[AGENT_DEBUG] _get_news: Fetched article: '{title}'")
                return f"Top news on '{topic}': {title}. {description}"
            elif data.get("status") == "error":
                 error_msg = data.get('message', 'Could not retrieve news.')
                 print(f"[AGENT_DEBUG] _get_news: NewsAPI error: {error_msg}")
                 return f"NewsAPI error for '{topic}': {error_msg}"
            else:
                print(f"[AGENT_DEBUG] _get_news: No news found for topic '{topic}'.")
                return f"No news found for topic '{topic}'."
        except Exception as e:
            print(f"[AGENT_DEBUG] ERROR _get_news: {e}")
            return f"Error fetching news: {e}"

    def _get_time(self, params):
        from datetime import datetime
        print(f"[AGENT_DEBUG] _get_time: Returning current time.")
        return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

In [19]:
LLM_API_KEY = "6ba1bc9d36c18b34a8e1a820d132b0b3614192ca291165e56799eff76f03304b" # This looks like a Mistral API key
agent = CustomSmartHomeAgent(llm_api_key=LLM_API_KEY)

print("Agent initialized.")

[AGENT_DEBUG] CustomSmartHomeAgent initialized. LLM lang: en
[AGENT_DEBUG] Initial device states: {'kitchen_lamp': {'status': 'off'}, 'bathroom_lamp': {'status': 'off', 'brightness': 75}, 'room_1_lamp': {'status': 'off'}, 'room_2_lamp': {'status': 'off', 'brightness': 75}, 'room_1_ac': {'status': 'off', 'temperature': 20}, 'kitchen_ac': {'status': 'off', 'temperature': 20}, 'living_room_tv': {'status': 'off'}}
Agent initialized.


In [14]:
# Cell 4: Voice Assistant Core Logic and Setup for Colab

# --- Wake Word ---
WAKE_WORD = "hey assistant" # Assuming English

# --- Whisper Model ---
print("Loading Whisper model...")
# Using "base.en" for English and faster performance in Colab
# You can change to "small.en", "medium.en" for more accuracy but slower speed
try:
    import torch
    WHISPER_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    whisper_model = whisper.load_model("base.en", device=WHISPER_DEVICE)
    print(f"Whisper model 'base.en' loaded on {WHISPER_DEVICE}.")
except Exception as e:
    print(f"Error loading Whisper model: {e}. Trying on CPU.")
    try:
        WHISPER_DEVICE = "cpu"
        whisper_model = whisper.load_model("base.en", device=WHISPER_DEVICE)
        print("Whisper model 'base.en' loaded on CPU.")
    except Exception as e_cpu:
        print(f"Could not load Whisper model on CPU either: {e_cpu}")
        whisper_model = None


# --- TTS Model ---
print("Loading TTS model...")
try:
    # Using your specified model. If this fails, common alternatives are:
    # "tts_models/en/ljspeech/vits" or "tts_models/en/vctk/vits" (multi-speaker)
    tts_model = TTS(model_name="tts_models/en/ljspeech/tacotron2-DDC", progress_bar=True, gpu=False)
    print("TTS model 'tts_models/en/ljspeech/fastspeech2' loaded.")
except Exception as e:
    print(f"Error loading TTS model 'tts_models/en/ljspeech/fastspeech2': {e}")
    print("Attempting to load a common alternative: 'tts_models/en/ljspeech/vits'")
    try:
        tts_model = TTS(model_name="tts_models/en/ljspeech/vits", progress_bar=True, gpu=False)
        print("TTS model 'tts_models/en/ljspeech/vits' loaded.")
    except Exception as e2:
        print(f"Could not load alternative TTS model: {e2}")
        tts_model = None

# --- Adapted STT function for Colab ---
def transcribe_colab_audio(audio_file_path, model):
    if not model:
        print("Whisper model not loaded. Cannot transcribe.")
        return ""
    if not audio_file_path or not os.path.exists(audio_file_path):
        print(f"Audio file not found: {audio_file_path}")
        return ""
    try:
        # Whisper's transcribe function can take a file path directly
        # Setting language to English based on wake word and TTS model
        result = model.transcribe(audio_file_path, language='en', fp16=False if WHISPER_DEVICE == "cpu" else True)
        return result['text']
    except Exception as e:
        print(f"Error during transcription: {e}")
        return ""

# --- Adapted TTS function for Colab ---
def speak_colab(text_to_speak, tts_instance, output_filename="colab_assistant_output.wav"):
    if not tts_instance:
        print("TTS model not loaded. Cannot speak.")
        return None
    if not text_to_speak:
        print("No text to speak.")
        return None
    try:
        print(f"Synthesizing speech for: '{text_to_speak}'")
        tts_instance.tts_to_file(text=text_to_speak, file_path=output_filename)
        print(f"Speech saved to {output_filename}")
        return output_filename
    except Exception as e:
        print(f"Error during TTS synthesis: {e}")
        return None

print("\nCore assistant functions defined and models loaded (if successful).")

Loading Whisper model...
Whisper model 'base.en' loaded on cpu.
Loading TTS model...
 > tts_models/en/ljspeech/tacotron2-DDC is already downloaded.
 > vocoder_models/en/ljspeech/hifigan_v2 is already downloaded.
 > Using model: Tacotron2
 > Setting up Audio Processor...
 | > sample_rate:22050
 | > resample:False
 | > num_mels:80
 | > log_func:np.log
 | > min_level_db:-100
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:20
 | > fft_size:1024
 | > power:1.5
 | > preemphasis:0.0
 | > griffin_lim_iters:60
 | > signal_norm:False
 | > symmetric_norm:True
 | > mel_fmin:0
 | > mel_fmax:8000.0
 | > pitch_fmin:1.0
 | > pitch_fmax:640.0
 | > spec_gain:1.0
 | > stft_pad_mode:reflect
 | > max_norm:4.0
 | > clip_norm:True
 | > do_trim_silence:True
 | > trim_db:60
 | > do_sound_norm:False
 | > do_amp_to_db_linear:True
 | > do_amp_to_db_mel:True
 | > do_rms_norm:False
 | > db_level:None
 | > stats_path:None
 | > base:2.718281828459045
 | > hop_length:256
 | > win_length:1024
 > Mo

In [6]:
!sudo apt-get install -y espeak-ng

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  espeak-ng-data libespeak-ng1 libpcaudio0 libsonic0
The following NEW packages will be installed:
  espeak-ng espeak-ng-data libespeak-ng1 libpcaudio0 libsonic0
0 upgraded, 5 newly installed, 0 to remove and 35 not upgraded.
Need to get 4,526 kB of archives.
After this operation, 11.9 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libpcaudio0 amd64 1.1-6build2 [8,956 B]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libsonic0 amd64 0.2.0-11build1 [10.3 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 espeak-ng-data amd64 1.50+dfsg-10ubuntu0.1 [3,956 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libespeak-ng1 amd64 1.50+dfsg-10ubuntu0.1 [207 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 espeak-ng amd64 1.50+dfsg-1

In [7]:
!pip install flask-ngrok > /dev/null 2>&1

In [8]:
!pip install flask-cors
!pip install pyngrok

In [6]:
import requests
from datetime import datetime # For agent's _get_time

# Standalone function for Isfahan weather for the panel
def fetch_isfahan_weather_standalone():
    location = 'isfahan'
    print(f"[DEBUG] fetch_isfahan_weather_standalone: Fetching weather for {location}")
    try:
        geo_resp = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": location, "count": 1, "language": "en", "format": "json"}
        )
        geo_resp.raise_for_status()
        geo_data = geo_resp.json()
        if "results" not in geo_data or not geo_data["results"]:
            print(f"[DEBUG] fetch_isfahan_weather_standalone: Could not find geo data for {location}")
            return f"N/A (geo)"

        result = geo_data["results"][0]
        lat = result["latitude"]
        lon = result["longitude"]

        weather_resp = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={"latitude": lat, "longitude": lon, "current_weather": True}
        )
        weather_resp.raise_for_status()
        weather_data = weather_resp.json()

        if "current_weather" in weather_data:
            temp = weather_data["current_weather"]["temperature"]
            print(f"[DEBUG] fetch_isfahan_weather_standalone: Fetched temp {temp}°C")
            return f"{temp}°C"
        else:
            print(f"[DEBUG] fetch_isfahan_weather_standalone: Could not get current weather data.")
            return f"N/A (weather)"
    except Exception as e:
        print(f"[DEBUG] ERROR in fetch_isfahan_weather_standalone: {e}")
        return "N/A (err)"

# Standalone function for the news panel
def fetch_panel_news_standalone(count=3):
    topic = "technology"
    api_key = "9a332f8925de495f81a0ca08b526283b" # From your notebook
    url = "https://newsapi.org/v2/everything"
    print(f"[DEBUG] fetch_panel_news_standalone: Fetching {count} news on {topic}")

    if not api_key or "YOUR_NEWS_API_KEY" in api_key:
        print("[DEBUG] WARNING fetch_panel_news_standalone: NewsAPI key is a placeholder or missing.")
        return {"error": "NewsAPI key not configured for panel."}
    try:
        response = requests.get(url, params={
            "q": topic, "language": "en", "sortBy": "publishedAt",
            "pageSize": count, "apiKey": api_key
        })
        response.raise_for_status()
        data = response.json()
        if data.get("status") == "ok" and data.get("articles"):
            articles_to_return = [
                {"title": article.get("title", "No title"),
                 "description": article.get("description", ""),
                 "url": article.get("url", "#")}
                for article in data["articles"]
            ]
            print(f"[DEBUG] fetch_panel_news_standalone: Fetched {len(articles_to_return)} articles.")
            return articles_to_return
        elif data.get("status") == "error":
            print(f"[DEBUG] fetch_panel_news_standalone: NewsAPI error: {data.get('message')}")
            return {"error": f"NewsAPI (panel): {data.get('message', 'Could not retrieve news.')}"}
        else:
            print(f"[DEBUG] fetch_panel_news_standalone: No news found for panel.")
            return {"error": "No news found for panel."}
    except Exception as e:
        print(f"[DEBUG] ERROR in fetch_panel_news_standalone: {e}")
        return {"error": f"Error fetching panel news: {str(e)}"}

In [ ]:
#  //<script src="https://cdn.jsdelivr.net/npm/@picovoice/porcupine-web@3.0.2/dist/iife/index.js"></script>
#     //<script src="https://cdn.jsdelivr.net/npm/@picovoice/web-voice-processor@4.0.2/dist/iife/index.js"></script>
#     //<script src="https://cdn.jsdelivr.net/npm/@picovoice/porcupine-web@3.0.2/dist/iife/porcupine_worker.js"></script>

In [ ]:

      # <script src="/content/static/js/porcupine.js"></script>

In [7]:
INDEX_HTML = """
<!DOCTYPE html>
<html lang="en" dir="ltr">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title id="pageTitle">Smart Home Assistant</title>
    <!-- No external voice libraries are needed for the Web Speech API approach -->
    <style>
        /* --- CSS Reset & Basic Styles --- */
        *, *::before, *::after { box-sizing: border-box; }
        body { font-family: 'Helvetica Neue', Arial, sans-serif; margin: 0; background-color: #121222; color: #E0E0E0; display: flex; justify-content: center; align-items: flex-start; min-height: 100vh; padding: 20px; }
        .app-container { width: 100%; max-width: 1200px; background-color: #1A1A2E; border-radius: 15px; box-shadow: 0 10px 30px rgba(0, 0, 0, 0.3); padding: 20px; }

        /* --- Header & Buttons --- */
        .app-header { display: flex; justify-content: space-between; align-items: center; padding-bottom: 20px; border-bottom: 1px solid #33334A; margin-bottom: 20px; flex-wrap: wrap; }
        .app-header .logo-title { display: flex; align-items: center; }
        .app-header .logo-title .home-icon { font-size: 28px; color: #7F5AF0; }
        .app-header .logo-title h1#mainHeading { font-size: 24px; margin: 0; color: #FFFFFF; }
        .app-header .logo-title p#subHeading { font-size: 12px; margin: 0; color: #A0A0B0; }
        .header-actions { display: flex; align-items: center; flex-wrap: wrap; gap: 10px; margin-top: 10px; }
        .header-button { background-color: #007bff; color: white; border: none; padding: 8px 15px; border-radius: 5px; cursor: pointer; font-size: 14px; transition: background-color 0.2s; }
        .header-button:hover:not(:disabled) { background-color: #0056b3; }
        .header-button:disabled { background-color: #555; cursor: not-allowed; opacity: 0.7; }
        .language-switcher { background-color: #7F5AF0; color: white; border: none; padding: 8px 15px; border-radius: 5px; cursor: pointer; font-size: 14px; }
        .language-switcher:hover { background-color: #9278F2; }
        .app-header .header-actions .status-dot { height: 10px; width: 10px; background-color: #34D399; border-radius: 50%; }
        .app-header .header-actions span#systemStatusText { font-size: 14px; color: #A0A0B0; }
        .app-header .header-actions .settings-icon { font-size: 24px; color: #A0A0B0; cursor: pointer; }

        /* --- Voice Assistant Panel & Animation --- */
        .voice-assistant-panel { background-color: #2A2A4E; border-radius: 10px; padding: 30px; text-align: center; margin-bottom: 30px; }
        #recordButton { background-color: #7F5AF0; border: none; border-radius: 50%; width: 70px; height: 70px; cursor: pointer; display: flex; justify-content: center; align-items: center; margin: 0 auto 15px auto; transition: background-color 0.3s ease, box-shadow 0.3s ease; box-shadow: 0 4px 15px rgba(0,0,0,0.2); position: relative; }
        #recordButton:disabled { background-color: #555; cursor: not-allowed; opacity: 0.6; }
        #recordButton.is-recording { background-color: #e74c3c; }
        #recordButton.is-recording::before { content: ''; position: absolute; top: 50%; left: 50%; width: 100%; height: 100%; background-color: transparent; border-radius: 50%; border: 3px solid rgba(255, 255, 255, 0.7); transform: translate(-50%, -50%) scale(1); opacity: 1; animation: waveAnimation 1.5s infinite ease-out; }
        @keyframes waveAnimation { 0% { transform: translate(-50%, -50%) scale(0.8); opacity: 1; border-width: 4px; } 70% { transform: translate(-50%, -50%) scale(1.5); opacity: 0; border-width: 1px; } 100% { transform: translate(-50%, -50%) scale(1.5); opacity: 0; border-width: 1px; } }
        .voice-assistant-panel .status-text#voiceAssistantStatus { font-size: 18px; color: #FFFFFF; margin-bottom: 5px; }
        .voice-assistant-panel .prompt-text#voiceAssistantPrompt { font-size: 13px; color: #A0A0B0; }

        /* --- General Layout and Components --- */
        .summary-dashboard { display: grid; grid-template-columns: repeat(auto-fit, minmax(150px, 1fr)); gap: 15px; margin-bottom: 30px; }
        .summary-card { background-color: #2A2A4E; border-radius: 8px; padding: 15px; text-align: center; }
        .summary-card .count { font-size: 28px; font-weight: bold; color: #FFFFFF; margin-bottom: 5px; }
        .summary-card .label { font-size: 12px; color: #A0A0B0; }
        .summary-card.temp .count { color: #FFD700; }
        .content-wrapper { display: flex; flex-wrap: wrap; gap: 20px; }
        .main-column { flex: 2; min-width: 300px; }
        .sidebar-column { flex: 1; min-width: 280px; }
        .device-controls h2 { font-size: 20px; color: #FFFFFF; margin-top: 0; margin-bottom: 15px; padding-bottom: 5px; border-bottom: 1px solid #33334A; }
        .device-grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(230px, 1fr)); gap: 15px; margin-bottom: 30px; }
        .device-card { background-color: #2A2A4E; border-radius: 8px; padding: 15px; display: flex; flex-direction: column; }
        .device-card .card-header { display: flex; justify-content: space-between; align-items: center; margin-bottom: 10px; }
        .device-card .card-header .icon-name { display: flex; align-items: center; }
        .device-card .icon { font-size: 24px; color: #7F5AF0; }
        .device-card .name { font-size: 16px; color: #FFFFFF; }
        .device-card .status-text-small { font-size: 12px; color: #A0A0B0; }
        .device-card .toggle-switch, .device-card input[type="range"], .device-card .ac-modes button, .device-card .temp-adjust button { pointer-events: none !important; opacity: 0.6 !important; cursor: not-allowed !important; }
        .device-card .toggle-switch input { pointer-events: none !important; }
        .toggle-switch { position: relative; display: inline-block; width: 50px; height: 26px; }
        .toggle-switch input { opacity: 0; width: 0; height: 0; }
        .toggle-slider { position: absolute; top: 0; left: 0; right: 0; bottom: 0; background-color: #444466; transition: .4s; border-radius: 26px; }
        .toggle-slider:before { position: absolute; content: ""; height: 18px; width: 18px; left: 4px; bottom: 4px; background-color: white; transition: .4s; border-radius: 50%; }
        input:checked + .toggle-slider { background-color: #7F5AF0; }
        input:checked + .toggle-slider:before { transform: translateX(24px); }
        .device-card .brightness-control, .device-card .temp-control-display { margin-top: 10px; }
        .device-card .brightness-control .brightness-percentage-text { font-weight: bold; color: #E0E0E0; }
        .device-card .ac-temp-display, .device-card .temp-display { font-size: 16px; color: #FFFFFF; font-weight: bold; text-align: center; margin: 10px 0; }
        .news-panel-styled { background-color: #2A2A4E; border-radius: 8px; padding: 20px; height: auto; max-height: 500px; overflow-y: auto; }
        .news-panel-styled h3#latestNewsHeading { font-family: 'Georgia', Times, 'Times New Roman', serif; color: #7F5AF0; margin-top: 0; margin-bottom: 15px; text-align: center; font-size: 1.5em; }
        #newsContent article { margin-bottom: 15px; border-bottom: 1px solid #3E3E6E; padding-bottom: 15px; }
        #newsContent article:last-child { border-bottom: none; padding-bottom: 0;}
        #newsContent article h4 { font-family: 'Helvetica Neue', Helvetica, Arial, sans-serif; color: #E0E0E0; font-size: 1.1em; margin: 0 0 8px 0; font-weight: 600; }
        #newsContent article h4 a { color: #E0E0E0; text-decoration: none; }
        #newsContent article h4 a:hover { color: #7F5AF0; text-decoration: underline; }
        #newsContent article p { font-family: 'Arial', sans-serif; color: #A0A0B0; font-size: 0.9em; line-height: 1.5; margin: 0; }
        #newsContent .loading-news, #newsContent .error-news { color: #A0A0B0; text-align: center; padding: 20px; }
        .interaction-area { margin-top: 30px; padding: 20px; background-color: #2A2A4E; border-radius: 10px; }
        .interaction-area h3#manualCommandHeading { margin-top:0; margin-bottom: 15px; color: #FFFFFF; }
        .input-group { display: flex; margin-bottom: 15px; }
        #commandInput { flex-grow: 1; border: 1px solid #3E3E6E; background-color: #1A1A2E; color: #E0E0E0; padding: 10px 15px; font-size: 14px; outline: none; }
        #sendButton { background-color: #7F5AF0; color: white; border: none; padding: 10px 20px; font-size: 14px; cursor: pointer; transition: background-color 0.3s ease; }
        #sendButton:hover { background-color: #9278F2; }
        .status, .response-area { padding: 10px; border-radius: 5px; font-size: 0.9em; margin-bottom: 10px; min-height: 18px; }
        .status { background-color: #3E3E6E; color: #A0A0B0; }
        .response-area { background-color: #3E3E6E; color: #E0E0E0; white-space: pre-wrap; }
        audio#responseAudio { display: none; }

        /* --- RTL/LTR Specific Styles --- */
        html[dir="ltr"] .app-header .logo-title .home-icon,
        html[dir="ltr"] .app-header .header-actions span#systemStatusText,
        html[dir="ltr"] .device-card .icon { margin-right: 10px; }
        html[dir="ltr"] .language-switcher { margin-left: 20px; }
        html[dir="ltr"] #commandInput { border-radius: 5px 0 0 5px; }
        html[dir="ltr"] #sendButton { border-radius: 0 5px 5px 0; }
        html[dir="ltr"] .response-area { text-align: left; }

        html[dir="rtl"] { font-family: 'Tahoma', 'Helvetica Neue', Arial, sans-serif; }
        html[dir="rtl"] .app-header .logo-title .home-icon,
        html[dir="rtl"] .app-header .header-actions span#systemStatusText,
        html[dir="rtl"] .device-card .icon { margin-left: 10px; }
        html[dir="rtl"] .language-switcher { margin-right: 20px; }
        html[dir="rtl"] #commandInput { border-radius: 0 5px 5px 0; }
        html[dir="rtl"] #sendButton { border-radius: 5px 0 0 5px; }
        html[dir="rtl"] .response-area { text-align: right; }
    </style>
</head>
<body>
    <div class="app-container">
        <header class="app-header">
            <div class="logo-title">
                <span class="home-icon">🏠</span>
                <div>
                    <h1 id="mainHeading">Smart Home Assistant</h1>
                    <p id="subHeading">Voice-controlled device orchestrator</p>
                </div>
            </div>
            <div class="header-actions">
                <button id="enableWakeWordButton" class="header-button" data-lang-key="enableHandsFree">Enable Hands-Free</button>
                <button id="languageSwitcher" class="language-switcher">فارسی</button>
                <span class="status-dot"></span>
                <span id="systemStatusText">Smart System</span>
                <span class="settings-icon">⚙️</span>
            </div>
        </header>

        <main class="main-content-area">
            <section class="voice-assistant-panel">
                <button id="recordButton" title="Record Voice"></button>
                <div class="status-text" id="voiceAssistantStatus">Voice Assistant Ready</div>
                <div class="prompt-text" id="voiceAssistantPrompt">Click button to activate. Or enable hands-free mode.</div>
            </section>

            <section class="summary-dashboard">
                <div class="summary-card"><div class="count" id="lampsActiveCount">0</div><div class="label" data-lang-key="lampsActive">Lamps Active</div></div>
                <div class="summary-card"><div class="count" id="acActiveCount">0</div><div class="label" data-lang-key="acUnitsOn">AC Units On</div></div>
                <div class="summary-card"><div class="count" id="tvActiveCount">0</div><div class="label" data-lang-key="tvActive">TV Active</div></div>
                <div class="summary-card temp"><div class="count" id="averageTempDisplay">--°C</div><div class="label" data-lang-key="isfahanTemp">Isfahan Temp</div></div>
            </section>

            <div class="content-wrapper">
                <div class="main-column">
                    <section class="device-controls">
                        <h2 data-lang-key="lightingControl">Lighting Control</h2>
                        <div class="device-grid">
                            <div class="device-card" data-device-key="kitchen_lamp"><div class="card-header"><div class="icon-name"><span class="icon">💡</span><div><div class="name" data-lang-key="kitchenLamp">Kitchen Lamp</div><div class="status-text-small">Off</div></div></div><label class="toggle-switch"><input type="checkbox" name="kitchen_lamp_toggle"><span class="toggle-slider"></span></label></div></div>
                            <div class="device-card" data-device-key="bathroom_lamp"><div class="card-header"><div class="icon-name"><span class="icon">💡</span><div><div class="name" data-lang-key="bathroomLamp">Bathroom Lamp</div><div class="status-text-small">Off</div></div></div><label class="toggle-switch"><input type="checkbox" name="bathroom_lamp_toggle"><span class="toggle-slider"></span></label></div><div class="brightness-control"><span data-lang-key="brightness">Brightness:</span> <span class="brightness-percentage-text">0%</span><input type="range" min="0" max="100" value="0"></div></div>
                            <div class="device-card" data-device-key="room_1_lamp"><div class="card-header"><div class="icon-name"><span class="icon">💡</span><div><div class="name" data-lang-key="room1Lamp">Room 1 Lamp</div><div class="status-text-small">Off</div></div></div><label class="toggle-switch"><input type="checkbox" name="room1_lamp_toggle"><span class="toggle-slider"></span></label></div></div>
                            <div class="device-card" data-device-key="room_2_lamp"><div class="card-header"><div class="icon-name"><span class="icon">💡</span><div><div class="name" data-lang-key="room2Lamp">Room 2 Lamp</div><div class="status-text-small">Off</div></div></div><label class="toggle-switch"><input type="checkbox" name="room2_lamp_toggle"><span class="toggle-slider"></span></label></div><div class="brightness-control"><span data-lang-key="brightness">Brightness:</span> <span class="brightness-percentage-text">0%</span><input type="range" min="0" max="100" value="0"></div></div>
                        </div>
                        <h2 data-lang-key="climateControl">Climate Control</h2>
                        <div class="device-grid">
                            <div class="device-card" data-device-key="room_1_ac"><div class="card-header"><div class="icon-name"><span class="icon">❄️</span><div><div class="name" data-lang-key="room1AC">Room 1 AC</div><div class="status-text-small">Off</div></div></div><label class="toggle-switch"><input type="checkbox" name="room1_ac_toggle"><span class="toggle-slider"></span></label></div><div class="temp-control-display"><div class="ac-temp-display">20°C</div></div></div>
                            <div class="device-card" data-device-key="kitchen_ac"><div class="card-header"><div class="icon-name"><span class="icon">❄️</span><div><div class="name" data-lang-key="kitchenAC">Kitchen AC</div><div class="status-text-small">Off</div></div></div><label class="toggle-switch"><input type="checkbox" name="kitchen_ac_toggle"><span class="toggle-slider"></span></label></div><div class="temp-control-display"><div class="temp-display">20°C</div></div></div>
                        </div>
                        <h2 data-lang-key="entertainmentControl">Entertainment</h2>
                        <div class="device-grid">
                            <div class="device-card" data-device-key="living_room_tv"><div class="card-header"><div class="icon-name"><span class="icon">📺</span><div><div class="name" data-lang-key="livingRoomTV">Living Room TV</div><div class="status-text-small">Off</div></div></div><label class="toggle-switch"><input type="checkbox" name="livingroom_tv_toggle"><span class="toggle-slider"></span></label></div></div>
                        </div>
                    </section>
                </div>
                <aside class="sidebar-column">
                    <div id="newsPanel" class="news-panel-styled">
                        <h3 id="latestNewsHeading" data-lang-key="latestNews">Latest News</h3>
                        <div id="newsContent" class="loading-news" data-lang-key="loadingNews">Loading news...</div>
                    </div>
                </aside>
            </div>
            <section class="interaction-area">
                 <h3 id="manualCommandHeading" data-lang-key="manualCommandArea">Manual Command & Agent Output</h3>
                <div class="input-group">
                    <input type="text" id="commandInput" placeholder="Or type your command here...">
                    <button id="sendButton" data-lang-key="sendButton">Send</button>
                </div>
                <div class="status" id="statusArea" data-lang-key="statusReady">Ready. Use the voice button or type a command.</div>
                <div class="response-area" id="agentResponseArea" data-lang-key="agentResponsePlaceholder">Agent's response will appear here.</div>
                <audio id="responseAudio"></audio>
            </section>
        </main>
    </div>

    <script>
        // --- START: Language and Translations ---
        const translations = {
            en: {
                pageTitle: "Smart Home Assistant", mainHeading: "Smart Home Assistant", subHeading: "Voice-controlled device orchestrator", languageSwitcher: "فارسی", systemStatusText: "Smart System", enableHandsFree: "Enable Hands-Free", disableHandsFree: "Disable Hands-Free",
                voiceAssistantStatus: "Voice Assistant Ready", voiceAssistantPrompt: "Click button to activate. Or enable hands-free mode.", recordButtonTitle: "Record Voice", stopRecordButtonTitle: "Stop Recording",
                lampsActive: "Lamps Active", acUnitsOn: "AC Units On", tvActive: "TV Active", isfahanTemp: "Isfahan Temp",
                lightingControl: "Lighting Control", kitchenLamp: "Kitchen Lamp", bathroomLamp: "Bathroom Lamp", room1Lamp: "Room 1 Lamp", room2Lamp: "Room 2 Lamp", brightness: "Brightness:",
                climateControl: "Climate Control", room1AC: "Room 1 AC", kitchenAC: "Kitchen AC", entertainmentControl: "Entertainment", livingRoomTV: "Living Room TV",
                latestNews: "Latest News", loadingNews: "Loading news...", errorLoadingNews: "Error loading news.", noNewsFound: "No news articles found.",
                manualCommandArea: "Manual Command & Agent Output", commandInputPlaceholder: "Or type your command here...", sendButton: "Send",
                statusReady: "Ready. Use the voice button or type a command.", statusRecording: "Recording... Click button again to stop.", statusProcessingAudio: "Processing audio...",
                statusNoAudioRecorded: "No audio recorded.", statusSendingCommand: "Sending command: ", statusMicError: "Error: Mic access denied. ",
                statusErrorSendingAudio: "Error sending audio.", statusErrorSendingCommand: "Error sending command.", agentResponsePlaceholder: "Agent's response will appear here.",
                statusAgentResponded: "Agent responded.", statusAgentError: "Error from agent: ", deviceStatusOn: "On", deviceStatusOff: "Off",
                statusHandsFreeActive: "Hands-free mode is active. Say 'Hey Assistant'.", statusInitializingEngine: "Initializing engine...", statusListeningForWakeWord: "Listening for 'Hey Assistant'...",
                statusWakeWordDetected: "Wake word detected! Listening for command...", statusListeningForCommand: "Listening for command...", statusEngineError: "Error: Web Speech API is not supported or failed."
            },
            fa: {
                pageTitle: "دستیار هوشمند خانه", mainHeading: "دستیار هوشمند خانه", subHeading: "ارکستراتور دستگاه با کنترل صوتی", languageSwitcher: "English", systemStatusText: "سیستم هوشمند", enableHandsFree: "فعال کردن دست‌آزاد", disableHandsFree: "غیرفعال کردن دست‌آزاد",
                voiceAssistantStatus: "دستیار صوتی آماده است", voiceAssistantPrompt: "دکمه را برای فعال‌سازی کلیک کنید. یا حالت دست‌آزاد را فعال کنید.", recordButtonTitle: "ضبط صدا", stopRecordButtonTitle: "توقف ضبط",
                lampsActive: "لامپ فعال", acUnitsOn: "کولر فعال", tvActive: "تلویزیون فعال", isfahanTemp: "دمای اصفهان",
                lightingControl: "کنترل روشنایی", kitchenLamp: "لامپ آشپزخانه", bathroomLamp: "لامپ حمام", room1Lamp: "لامپ اتاق ۱", room2Lamp: "لامپ اتاق ۲", brightness: "روشنایی:",
                climateControl: "کنترل تهویه", room1AC: "کولر اتاق ۱", kitchenAC: "کولر آشپزخانه", entertainmentControl: "کنترل سرگرمی", livingRoomTV: "تلویزیون اتاق نشیمن",
                latestNews: "آخرین اخبار", loadingNews: "در حال بارگذاری اخبار...", errorLoadingNews: "خطا در بارگذاری اخبار.", noNewsFound: "موردی برای نمایش یافت نشد.",
                manualCommandArea: "فرمان دستی و خروجی دستیار", commandInputPlaceholder: "یا فرمان خود را اینجا تایپ کنید...", sendButton: "ارسال",
                statusReady: "آماده. از دکمه ضبط صدا استفاده کنید یا فرمان تایپ کنید.", statusRecording: "در حال ضبط... برای توقف دوباره کلیک کنید.", statusProcessingAudio: "در حال پردازش صدا...",
                statusNoAudioRecorded: "صدایی ضبط نشده است.", statusSendingCommand: "در حال ارسال فرمان: ", statusMicError: "خطا: دسترسی به میکروفون امکان‌پذیر نیست. ",
                statusErrorSendingAudio: "خطا در ارسال صدا.", statusErrorSendingCommand: "خطا در ارسال فرمان.", agentResponsePlaceholder: "پاسخ دستیار در اینجا نمایش داده می‌شود.",
                statusAgentResponded: "ایجنت پاسخ داد.", statusAgentError: "خطا از ایجنت: ", deviceStatusOn: "روشن", deviceStatusOff: "خاموش",
                statusHandsFreeActive: "حالت دست‌آزاد فعال است. بگویید 'هی اسیستنت'.", statusInitializingEngine: "در حال آماده‌سازی موتور...", statusListeningForWakeWord: "در حال شنیدن برای 'هی اسیستنت'...",
                statusWakeWordDetected: "کلمه فعال‌سازی شنیده شد! در حال شنیدن فرمان...", statusListeningForCommand: "در حال شنیدن فرمان...", statusEngineError: "خطا: Web Speech API پشتیبانی نمی‌شود یا ناموفق بود."
            }
        };
        let currentLang = localStorage.getItem('preferredLang') || 'en';
        let lastFetchedDeviceStates = null;

        const commandInput = document.getElementById('commandInput');
        const recordButton = document.getElementById('recordButton');
        const sendButton = document.getElementById('sendButton');
        const statusArea = document.getElementById('statusArea');
        const agentResponseArea = document.getElementById('agentResponseArea');
        const responseAudio = document.getElementById('responseAudio');
        const averageTempDisplay = document.getElementById('averageTempDisplay');
        const newsContentDiv = document.getElementById('newsContent');
        const languageSwitcherButton = document.getElementById('languageSwitcher');
        const enableWakeWordButton = document.getElementById('enableWakeWordButton');
        const lampsActiveCountEl = document.getElementById('lampsActiveCount');
        const acActiveCountEl = document.getElementById('acActiveCount');
        const tvActiveCountEl = document.getElementById('tvActiveCount');

        let mediaRecorder;
        let audioChunks = [];
        let isPushToTalkRecording = false;
        let isWakeWordModeActive = false;
        let recognition;

        function setLanguage(lang) {
            currentLang = lang;
            localStorage.setItem('preferredLang', lang);
            document.documentElement.lang = lang;
            document.documentElement.dir = lang === 'fa' ? 'rtl' : 'ltr';
            document.querySelectorAll('[data-lang-key]').forEach(element => {
                const key = element.getAttribute('data-lang-key');
                if (translations[lang]?.[key]) {
                    element.textContent = translations[lang][key];
                }
            });
            const elementsById = { pageTitle: 'pageTitle', mainHeading: 'mainHeading', subHeading: 'subHeading', languageSwitcher: 'languageSwitcher', systemStatusText: 'systemStatusText', voiceAssistantStatus: 'voiceAssistantStatus', voiceAssistantPrompt: 'voiceAssistantPrompt', latestNewsHeading: 'latestNews', manualCommandHeading: 'manualCommandHeading', enableWakeWordButton: isWakeWordModeActive ? 'disableHandsFree' : 'enableHandsFree' };
            for (const idKey in elementsById) {
                const element = document.getElementById(idKey);
                if (element && translations[lang]?.[elementsById[idKey]]) {
                    element.textContent = translations[lang][elementsById[idKey]];
                }
            }
            if(recordButton) recordButton.title = translations[lang].recordButtonTitle;
            if(commandInput) commandInput.placeholder = translations[lang].commandInputPlaceholder;
            const newsStatusEl = document.querySelector('#newsContent.loading-news, #newsContent.error-news');
            if (newsStatusEl?.classList.contains('loading-news')) newsStatusEl.textContent = translations[lang].loadingNews;
            else if (newsStatusEl?.classList.contains('error-news')) newsStatusEl.textContent = translations[lang].errorLoadingNews;
            if (lastFetchedDeviceStates) updateDeviceUI(lastFetchedDeviceStates);
        }

        document.addEventListener('DOMContentLoaded', () => {
            if (!commandInput || !recordButton || !sendButton || !statusArea || !agentResponseArea || !responseAudio || !averageTempDisplay || !newsContentDiv || !languageSwitcherButton || !enableWakeWordButton) {
                console.error("One or more critical UI elements could not be found. Check HTML IDs.");
                if (statusArea) statusArea.textContent = "UI Error: Elements missing.";
                return;
            }
            window.SpeechRecognition = window.SpeechRecognition || window.webkitSpeechRecognition;
            if (!window.SpeechRecognition) {
                console.error("Web Speech API is not supported in this browser.");
                enableWakeWordButton.disabled = true;
                enableWakeWordButton.textContent = "Hands-Free Not Supported";
            }

            setLanguage(currentLang);

            // 🟢 Fetch the latest device states after page load
              fetch('/get_initial_states')
                  .then(response => response.json())
                  .then(data => {
                      if (data.device_states) {
                          updateDeviceUI(data.device_states);
                      }
                  })
                  .catch(error => {
                      console.error("Error fetching initial device states:", error);
                  });


            fetchPanelWeather();
            fetchPanelNews();
            enableWakeWordButton.addEventListener('click', toggleWakeWordMode);
            recordButton.onclick = handleRecordButtonClick;
            sendButton.onclick = handleSendButtonClick;
            languageSwitcherButton.addEventListener('click', () => {
                setLanguage(currentLang === 'en' ? 'fa' : 'en');
            });
            commandInput.addEventListener('keypress', (e) => { if (e.key === 'Enter') handleSendButtonClick(); });



        });

        function toggleWakeWordMode() {
            if (!window.SpeechRecognition) {
                statusArea.textContent = translations[currentLang].statusEngineError;
                return;
            }
            isWakeWordModeActive = !isWakeWordModeActive;
            enableWakeWordButton.textContent = isWakeWordModeActive ? translations[currentLang].disableHandsFree : translations[currentLang].enableHandsFree;
            recordButton.disabled = isWakeWordModeActive;

            if (isWakeWordModeActive) {
                listenForWakeWord();
            } else {
                if (recognition) {
                    recognition.stop();
                    recognition = null;
                }
                statusArea.textContent = translations[currentLang].statusReady;
            }
        }

        function listenForWakeWord() {
            if (!isWakeWordModeActive) return;

            statusArea.textContent = translations[currentLang].statusListeningForWakeWord;
            recognition = new window.SpeechRecognition();
            recognition.lang = 'en-US';
            recognition.interimResults = false;
            recognition.continuous = false;
            let isTransitioningToCommand = false;

            recognition.onresult = (event) => {
                const transcript = event.results[0][0].transcript.toLowerCase();
                console.log("Heard:", transcript);
                if (transcript.includes("hey assistant")) {
                    isTransitioningToCommand = true;
                    statusArea.textContent = translations[currentLang].statusWakeWordDetected;
                    recordButton.classList.add('is-recording');
                    listenForCommand();
                }
            };

            recognition.onerror = (e) => {
                console.error("Wake word recognition error:", e.error);
            };

            recognition.onend = () => {
                if (isWakeWordModeActive && !isTransitioningToCommand) {
                    setTimeout(() => listenForWakeWord(), 100);
                }
            };

            try {
                recognition.start();
            } catch (e) {
                console.error("Error starting wake word recognition:", e);
            }
        }

        function listenForCommand() {
            if (!isWakeWordModeActive) {
                 recordButton.classList.remove('is-recording');
                 return;
            }

            statusArea.textContent = translations[currentLang].statusListeningForCommand;
            const commandRecognition = new window.SpeechRecognition();
            commandRecognition.lang = 'en-US';
            commandRecognition.interimResults = false;
            commandRecognition.continuous = false;
            let commandHeard = false;

            const timeoutId = setTimeout(() => {
                if (!commandHeard) {
                    console.log("Command listener timed out after 7 seconds.");
                    commandRecognition.abort();
                }
            }, 7000);

            commandRecognition.onresult = (event) => {
                clearTimeout(timeoutId);
                commandHeard = true;
                const command = event.results[0][0].transcript;
                console.log("Command heard:", command);
                commandInput.value = command;
                handleSendButtonClick();
            };

            commandRecognition.onerror = (e) => {
                clearTimeout(timeoutId);
                console.error("Command recognition error:", e.error);
                if (e.error !== 'aborted') {
                    statusArea.textContent = "Didn't catch that. Listening for wake word again.";
                }
            };

            commandRecognition.onend = () => {
                clearTimeout(timeoutId);
                recordButton.classList.remove('is-recording');
                if (isWakeWordModeActive) {
                    console.log("Command recognition finished. Restarting wake word listener.");
                    listenForWakeWord();
                }
            };

            try {
                commandRecognition.start();
            } catch (e) {
                clearTimeout(timeoutId);
                console.error("Error starting command recognition:", e);
                recordButton.classList.remove('is-recording');
                if (isWakeWordModeActive) listenForWakeWord();
            }
        }

        async function handleRecordButtonClick() {
            if (isWakeWordModeActive) {
                statusArea.textContent = translations[currentLang].statusHandsFreeActive;
                return;
            }
            if (!isPushToTalkRecording) {
                try {
                    const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
                    mediaRecorder = new MediaRecorder(stream, { mimeType: 'audio/webm' });
                    mediaRecorder.ondataavailable = event => { if (event.data.size > 0) audioChunks.push(event.data); };
                    mediaRecorder.onstop = () => sendAudioToBackend('push-to-talk');
                    audioChunks = []; mediaRecorder.start(); isPushToTalkRecording = true;
                    statusArea.textContent = translations[currentLang].statusRecording;
                    recordButton.classList.add('is-recording');
                    recordButton.title = translations[currentLang].stopRecordButtonTitle;
                } catch (err) {
                    isPushToTalkRecording = false; recordButton.classList.remove('is-recording'); recordButton.title = translations[currentLang].recordButtonTitle;
                    console.error("Mic error:", err); statusArea.textContent = translations[currentLang].statusMicError + err.message;
                }
            } else {
                if (mediaRecorder && mediaRecorder.state === "recording") { mediaRecorder.stop(); }
            }
        }

        async function handleSendButtonClick() {
            const command = commandInput.value.trim();
            if (!command) {
                statusArea.textContent = currentLang === 'fa' ? "لطفاً یک فرمان تایپ کنید." : "Please type a command.";
                return;
            }
            statusArea.textContent = translations[currentLang].statusSendingCommand + command;
            agentResponseArea.textContent = currentLang === 'fa' ? 'در حال پردازش...' : 'Processing...';
            responseAudio.src = '';
            try {
                const response = await fetch('/process_text_command', {
                    method: 'POST', headers: { 'Content-Type': 'application/json' },
                    body: JSON.stringify({ command: command })
                });
                const result = await response.json();
                handleBackendResponse(result);
            } catch (error) {
                console.error('Error sending text:', error);
                statusArea.textContent = translations[currentLang].statusErrorSendingCommand;
                agentResponseArea.textContent = 'Error: ' + error.message;
            }
        }

        async function sendAudioToBackend(source) {
            if (source !== 'push-to-talk') return;
            statusArea.textContent = translations[currentLang].statusProcessingAudio;
            isPushToTalkRecording = false;
            recordButton.classList.remove('is-recording');
            recordButton.title = translations[currentLang].recordButtonTitle;

            if (audioChunks.length === 0) {
                statusArea.textContent = translations[currentLang].statusNoAudioRecorded;
                return;
            }
            const audioBlob = new Blob(audioChunks, { type: 'audio/webm' });
            audioChunks = [];
            const formData = new FormData();
            formData.append('audio_data', audioBlob, 'recorded_audio.webm');
            try {
                const response = await fetch('/process_audio_command', { method: 'POST', body: formData });
                const result = await response.json();
                handleBackendResponse(result);
            } catch (error) {
                console.error(`Error sending audio from ${source}:`, error);
                statusArea.textContent = translations[currentLang].statusErrorSendingAudio;
            }
        }

        function handleBackendResponse(result) {
            console.log("Backend response received:", result);
            if (result.error) {
                statusArea.textContent = translations[currentLang].statusAgentError + (result.text_response || result.error);
                agentResponseArea.textContent = result.text_response || result.error;
            } else {
                statusArea.textContent = translations[currentLang].statusAgentResponded;
                agentResponseArea.textContent = result.text_response || (currentLang === 'fa' ? 'فرمان پردازش شد.' : 'Command processed.');
                 if (result.intent === 'news') {
                      fetchPanelNews(); // 🟢 update the news panel live
                   }

                if (result.transcribed_text) { commandInput.value = result.transcribed_text; }
            }
            if (result.audio_url) {
                responseAudio.src = result.audio_url + '?t=' + new Date().getTime();
                responseAudio.play().catch(e => console.warn("Audio play failed:", e));
            } else { responseAudio.src = ''; }
            if (result.device_states) { updateDeviceUI(result.device_states); }
        }

        function updateDeviceUI(deviceStates) {
            if (!deviceStates) return;
            lastFetchedDeviceStates = deviceStates;
            console.log("Updating UI with states:", deviceStates);
            for (const deviceKey in deviceStates) {
                const stateInfo = deviceStates[deviceKey];
                const cardElement = document.querySelector(`.device-card[data-device-key="${deviceKey}"]`);
                if (cardElement) {
                    const toggleInput = cardElement.querySelector('.toggle-switch input[type="checkbox"]');
                    const statusTextElement = cardElement.querySelector('.status-text-small');
                    if (toggleInput) toggleInput.checked = (stateInfo.status === "on");
                    if (statusTextElement) {
                        let currentStatusText = (stateInfo.status === "on") ? translations[currentLang].deviceStatusOn : translations[currentLang].deviceStatusOff;
                        if (deviceKey.includes("_ac") && typeof stateInfo.temperature !== 'undefined') {
                            const statusPart = (stateInfo.status === "on") ? translations[currentLang].deviceStatusOn : translations[currentLang].deviceStatusOff;
                            currentStatusText = `${statusPart} - ${stateInfo.temperature}°C`;
                            const mainAcTempDisplay = cardElement.querySelector('.ac-temp-display, .temp-display');
                            if (mainAcTempDisplay) mainAcTempDisplay.textContent = `${stateInfo.temperature}°C`;
                        }
                        if (deviceKey.includes("_lamp") && typeof stateInfo.brightness !== 'undefined') {
                            const brightnessTextSpan = cardElement.querySelector('.brightness-percentage-text');
                            if (brightnessTextSpan) brightnessTextSpan.textContent = `${stateInfo.brightness}%`;
                            const brightnessSlider = cardElement.querySelector('input[type="range"]');
                            if (brightnessSlider) brightnessSlider.value = stateInfo.brightness;
                            if (stateInfo.status === "on") {
                                currentStatusText = `${translations[currentLang].deviceStatusOn} - ${stateInfo.brightness}%`;
                            } else {
                                currentStatusText = `${translations[currentLang].deviceStatusOff} (${stateInfo.brightness}%)`;
                            }
                        }
                        statusTextElement.textContent = currentStatusText;
                    }
                }
            }
            updateSummaryDashboard(deviceStates);
        }

        function updateSummaryDashboard(deviceStates) {
            let lampsOn = 0; let acOn = 0; let tvOn = 0;
            for (const key in deviceStates) {
                if (deviceStates[key].status === "on") {
                    if (key.includes("_lamp")) lampsOn++;
                    else if (key.includes("_ac")) acOn++;
                    else if (key.includes("_tv")) tvOn++;
                }
            }
            if(lampsActiveCountEl) lampsActiveCountEl.textContent = lampsOn;
            if(acActiveCountEl) acActiveCountEl.textContent = acOn;
            if(tvActiveCountEl) tvActiveCountEl.textContent = tvOn;
        }

        async function fetchPanelWeather() {
            try {
                const response = await fetch('/get_panel_weather');
                const data = await response.json();
                if (data.error || !data.weather_string || data.weather_string.includes("N/A")) {
                    averageTempDisplay.textContent = currentLang === 'fa' ? 'خطا' : 'Error';
                } else { averageTempDisplay.textContent = data.weather_string; }
            } catch (error) {
                averageTempDisplay.textContent = currentLang === 'fa' ? 'خطا' : 'Error';
            }
        }

        async function fetchPanelNews() {
            newsContentDiv.innerHTML = `<p class="loading-news">${translations[currentLang].loadingNews}</p>`;
            try {
                const response = await fetch('/get_panel_news');
                const data = await response.json();
                if (data.error || !data.articles || data.articles.length === 0) {
                    newsContentDiv.innerHTML = `<p class="error-news">${data.error ? translations[currentLang].errorLoadingNews : translations[currentLang].noNewsFound}</p>`;
                    return;
                }
                let newsHTML = '';
                data.articles.forEach(article => {
                    const title = article.title || (currentLang === 'fa' ? "بدون عنوان" : "No title");
                    const description = article.description || (currentLang === 'fa' ? "توضیحات موجود نیست." : "No description available.");
                    const url = article.url || "#";
                    newsHTML += `<article><h4><a href="${url}" target="_blank" rel="noopener noreferrer">${title}</a></h4><p>${description}</p></article>`;
                });
                newsContentDiv.innerHTML = newsHTML;
            } catch (error) {
                newsContentDiv.innerHTML = `<p class="error-news">${translations[currentLang].errorLoadingNews}</p>`;
            }
        }
    </script>
</body>
</html>


"""

In [16]:
pc_url = " https://6650-143-14-203-35.ngrok-free.app/ping"

In [ ]:
from logging import debug
from flask import Flask, request, jsonify, send_from_directory, render_template_string
from flask_cors import CORS
from pyngrok import conf, ngrok
import os
import threading
from base64 import b64decode
import uuid # For unique filenames

# --- Configuration ---
# IMPORTANT: Set your ngrok authtoken if you have one for longer sessions/more features
conf.get_default().auth_token = "2y2KflemXHMMlr2b0TAspzSXNxl_7PWkPYfJ4iL8DefL1TQUd" # Replace with your token if you have one

# Create a directory to store audio responses if it doesn't exist
AUDIO_OUTPUT_DIR = 'static/audio_outputs'
if not os.path.exists(AUDIO_OUTPUT_DIR):
    os.makedirs(AUDIO_OUTPUT_DIR)

app = Flask(__name__ )
app.debug = True
CORS(app) # Enable CORS for all routes


@app.route('/')
def index():
    print("[FLASK_DEBUG] Serving index page.")
    return render_template_string(INDEX_HTML) # INDEX_HTML must be defined



@app.route('/get_initial_states', methods=['GET'])
def get_initial_states_route():
    """
    This new endpoint provides the initial state of all devices
    to the frontend when it first loads.
    """
    print(f"[FLASK_DEBUG] /get_initial_states called. Returning current agent states.")
    # The 'agent' object is global, so its current state is accessible here.
    return jsonify({'device_states': agent.status()})



@app.route('/get_panel_weather', methods=['GET'])
def get_panel_weather_route():
    print("[FLASK_DEBUG] /get_panel_weather called")
    try:
        weather_info_text = fetch_isfahan_weather_standalone()
        print(f"[FLASK_DEBUG] /get_panel_weather: Result: {weather_info_text}")
        return jsonify({'weather_string': weather_info_text})
    except Exception as e:
        print(f"[FLASK_DEBUG] ERROR in /get_panel_weather: {e}")
        return jsonify({'error': str(e), 'weather_string': 'N/A'}), 500

@app.route('/get_panel_news', methods=['GET'])
def get_panel_news_route():
    print("[FLASK_DEBUG] /get_panel_news called")
    try:
        news_data = fetch_panel_news_standalone(count=1)
        print(f"[FLASK_DEBUG] /get_panel_news: Result (type {type(news_data)}): {news_data}")
        if isinstance(news_data, dict) and "error" in news_data:
             return jsonify({'error': news_data["error"], 'articles': []}), 500
        return jsonify({'articles': news_data})
    except Exception as e:
        print(f"[FLASK_DEBUG] ERROR in /get_panel_news: {e}")
        return jsonify({'error': str(e), 'articles': []}), 500

@app.route('/process_text_command', methods=['POST'])
def handle_text_command():
    data = request.get_json()
    command = data.get('command')
    print(f"[FLASK_DEBUG] /process_text_command: Received command: '{command}'")

    # Initialize defaults
    text_response_for_json = "No command received."
    audio_file_url = None
    error_msg_for_json = None
    current_device_states = {} # Default to empty dict

    if command:
        try:
            text_from_agent_part = agent.execute_command(command) # This is the single combined string
            print(f"[FLASK_DEBUG] Will now call agent.status()")
            text_response_for_json = text_from_agent_part
            print(f"[FLASK_DEBUG] /process_text_commtext_response_for_jsontates : '{text_response_for_json}' (Type: {type(text_response_for_json)})")
            current_device_states = agent.status()
            print(f"[FLASK_DEBUG] /process_text_command: Sending to current_device_states : '{current_device_states}' (Type: {type(current_device_states)})")
            # current_device_states = json.dumps(string)
            # if string :
            if "lamp" in  command or "Lamp" in command:
                      url = pc_url
                      resp = requests.post(url, json={"source": "colab", "message": text_response_for_json})

            if text_response_for_json and agent.original_lan == 'en':

                  unique_filename = f"{str(uuid.uuid4())}.wav"
                  output_path = os.path.join(AUDIO_OUTPUT_DIR, unique_filename)
                  if tts_model:
                      # This print is crucial for the TTS part
                      print(f"[FLASK_DEBUG] /process_text_command: Sending to TTS model: '{text_response_for_json}' (Type: {type(text_response_for_json)})")
                      tts_model.tts_to_file(text=text_response_for_json, file_path=output_path)
                      if os.path.exists(output_path):
                          audio_file_url = f"/audio/{unique_filename}"
                          print(f"[FLASK_DEBUG] /process_text_command: TTS audio generated: {audio_file_url}")
                      else:
                          print(f"[FLASK_DEBUG] /process_text_command: TTS audio file NOT generated at {output_path}")
                  else:
                      print("[FLASK_DEBUG] /process_text_command: TTS model not available.")
            else:
                  print(f"[FLASK_DEBUG] /process_text_command: text_response_for_json is empty or None after splitting.")


        except Exception as e:
            print(f"[FLASK_DEBUG] !!! ERROR in /process_text_command try block: {e}")
            import traceback
            traceback.print_exc()

            error_msg_for_json = str(e)
            text_response_for_json = f"Error processing command in Flask: {e}"
            current_device_states = agent.device_states # Fallback to current agent state on major error
            print(f"[FLASK_DEBUG] In except block, text_response_for_json: '{text_response_for_json}'")
            print(f"[FLASK_DEBUG] In except block, current_device_states: {current_device_states}")



    if "news" in command or "what's the news" in command or "latest news" in command:
                                    print(f"[FLASK_DEBUG] Recognized 'news' intent from: {command}")
                                    response_payload = {
                                        "text_response": text_response_for_json,
                                        "intent": "news",
                                        'audio_url': audio_file_url,
                                         'device_states': current_device_states,
                                        'error': error_msg_for_json
                                    }
                                    return jsonify(response_payload)

    response_payload = {
        'text_response': text_response_for_json,
        'audio_url': audio_file_url,
        'device_states': current_device_states,
        'error': error_msg_for_json
    }
    # This print will show exactly what's going into jsonify
    print(f"[FLASK_DEBUG] /process_text_command: Preparing to return JSON. text_response type: {type(text_response_for_json)}, device_states type: {type(current_device_states)}")
    print(f"[FLASK_DEBUG] /process_text_command: Full response_payload: {response_payload}")
    return jsonify(response_payload)




@app.route('/process_audio_command', methods=['POST'])
def handle_audio_command():
    print("[FLASK_DEBUG] /process_audio_command: Received audio request.")
    text_response_for_json = "No audio received or error in processing."
    transcribed_text_for_ui = ""
    audio_file_url = None
    error_msg = None
    current_device_states = agent.device_states

    if 'audio_data' not in request.files:
        print("[FLASK_DEBUG] /process_audio_command: No audio file part in request.")
        return jsonify({'error': 'No audio file part.', 'device_states': current_device_states, 'text_response': 'Error: No audio data.'}), 400

    file = request.files['audio_data']
    if file.filename == '':
        print("[FLASK_DEBUG] /process_audio_command: No selected audio file.")
        return jsonify({'error': 'No selected audio file.', 'device_states': current_device_states, 'text_response': 'Error: No file selected.'}), 400

    if file:
        temp_audio_path = os.path.join(AUDIO_OUTPUT_DIR, f"temp_{str(uuid.uuid4())}.webm")
        print(f"[FLASK_DEBUG] /process_audio_command: Saving temp audio to {temp_audio_path}")
        file.save(temp_audio_path)
        try:
            print(f"[FLASK_DEBUG] /process_audio_command: Transcribing audio file: {temp_audio_path}")
            transcribed_text = transcribe_colab_audio(temp_audio_path, whisper_model)
            transcribed_text_for_ui = transcribed_text
            print(f"[FLASK_DEBUG] /process_audio_command: Transcribed text: '{transcribed_text}'")
            command_to_agent = transcribed_text

            if command_to_agent:
                text_for_speech_and_json = agent.execute_command(command_to_agent)

                text_response_for_json = text_for_speech_and_json
                current_device_states = agent.status()
                print(f"[FLASK_DEBUG] /process_audio_command: Agent text response: '{text_response_for_json}'")
                print(f"[FLASK_DEBUG] /process_audio_command: Agent updated states: {current_device_states}")
                if "lamp" in  command_to_agent or "Lamp" in command_to_agent:
                      url = pc_url
                      resp = requests.post(url, json={"source": "colab", "message": text_for_speech_and_json})
                if text_for_speech_and_json  and agent.original_lan == 'en':

                    unique_filename = f"{str(uuid.uuid4())}.wav"
                    output_path = os.path.join(AUDIO_OUTPUT_DIR, unique_filename)
                    if tts_model:
                        print(f"[FLASK_DEBUG] /process_audio_command: Sending to TTS model: '{text_for_speech_and_json}' (Type: {type(text_for_speech_and_json)})")
                        tts_model.tts_to_file(text=text_for_speech_and_json, file_path=output_path)
                        if os.path.exists(output_path):
                            audio_file_url = f"/audio/{unique_filename}"
                            print(f"[FLASK_DEBUG] /process_audio_command: TTS audio generated: {audio_file_url}")

                        else:
                            print(f"[FLASK_DEBUG] /process_audio_command: TTS audio file NOT generated at {output_path}")
                    else:
                        print("[FLASK_DEBUG] /process_audio_command: TTS model not available.")
            else:
                text_response_for_json = "Transcription resulted in empty command."
                print("[FLASK_DEBUG] /process_audio_command: Empty command after transcription.")
        except Exception as e:
            error_msg = str(e)
            text_response_for_json = f"Error processing audio in Flask: {e}"
            print(f"[FLASK_DEBUG] ERROR in /process_audio_command: {e}")
        finally:
            if os.path.exists(temp_audio_path):
                print(f"[FLASK_DEBUG] /process_audio_command: Removing temp audio file {temp_audio_path}")
                os.remove(temp_audio_path)

    if "news" in transcribed_text or "what's the news" in transcribed_text or "latest news" in transcribed_text:
                                    print(f"[FLASK_DEBUG] Recognized 'news' intent from: {transcribed_text}")
                                    response_payload = {
                                        "text_response": text_response_for_json,
                                        "intent": "news",
                                        'transcribed_text': transcribed_text_for_ui,
                                        'audio_url': audio_file_url,
                                         'device_states': current_device_states,
                                        'error': error_msg
                                    }
                                    return jsonify(response_payload)
    response_payload = {
        'text_response': text_response_for_json,
        'transcribed_text': transcribed_text_for_ui,
        'audio_url': audio_file_url,
        'device_states': current_device_states,
        'error': error_msg
    }
    print(f"[FLASK_DEBUG] /process_audio_command: Returning JSON: {response_payload}")
    return jsonify(response_payload)

@app.route('/audio/<filename>')
def serve_audio(filename):
    print(f"[FLASK_DEBUG] Serving audio file: {filename}")
    return send_from_directory(AUDIO_OUTPUT_DIR, filename)


# --- Start Flask app and ngrok tunnel ---
# Kill any existing ngrok tunnels/Flask apps if you rerun this cell
try:
    if 'public_url' in globals() and ngrok.get_tunnels():
        print(f"Closing existing ngrok tunnel: {public_url}")
        ngrok.disconnect(public_url)
        ngrok.kill()
    # You might need a way to stop the Flask app thread if it's already running
    # This is a bit tricky in Colab; often restarting the runtime is easiest for a clean slate.
except Exception as e:
    print(f"Error managing existing ngrok: {e}")

# Start ngrok tunnel
try:
    # Make sure pyngrok is configured with your auth token if you have one
    # conf.get_default().auth_token = "YOUR_NGROK_AUTHTOKEN" # Run this if not done globally
    public_url = ngrok.connect(5000) # Flask default port is 5000
    print(f" * ngrok tunnel \"{public_url}\" -> \"http://127.0.0.1:5000\"")
    print(f" * Access your UI at: {public_url}")

    # Start Flask app in a separate thread
    # Use a daemon thread so it automatically closes when the Colab cell/runtime stops
    threading.Thread(target=app.run, kwargs={'host':'0.0.0.0', 'port':5000, 'debug':True, 'use_reloader':False}, daemon=True).start()
    print("Flask server started. Visit the public URL above.")
    while True:
      pass  # <-- Keeps the Colab cell running
except Exception as e:
    print(f"Error starting ngrok or Flask: {e}")
    print("Please ensure you have ngrok installed and configured if necessary (e.g., with an authtoken for stable URLs).")
    print("You might need to provide your ngrok authtoken via pyngrok.conf.get_default().auth_token = 'YOUR_TOKEN'")

 * ngrok tunnel "NgrokTunnel: "https://086e-34-16-215-133.ngrok-free.app" -> "http://localhost:5000"" -> "http://127.0.0.1:5000"
 * Access your UI at: NgrokTunnel: "https://086e-34-16-215-133.ngrok-free.app" -> "http://localhost:5000"
Flask server started. Visit the public URL above.
 * Serving Flask app '__main__'
 * Debug mode: on


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.
INFO:werkzeug:127.0.0.1 - - [08/Jun/2025 15:04:03] "GET / HTTP/1.1" 200 -


[FLASK_DEBUG] Serving index page.


INFO:werkzeug:127.0.0.1 - - [08/Jun/2025 15:04:04] "GET /get_initial_states HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [08/Jun/2025 15:04:04] "GET /favicon.ico HTTP/1.1" 404 -


[FLASK_DEBUG] /get_panel_news called
[DEBUG] fetch_panel_news_standalone: Fetching 1 news on technology
[FLASK_DEBUG] /get_panel_weather called
[DEBUG] fetch_isfahan_weather_standalone: Fetching weather for isfahan
[FLASK_DEBUG] /get_initial_states called. Returning current agent states.
[AGENT_DEBUG] status:Coming
[AGENT_DEBUG] status: Returning dictionary_as_string: {"kitchen_lamp": {"status": "off"}, "bathroom_lamp": {"status": "off", "brightness": 75}, "room_1_lamp": {"status": "off"}, "room_2_lamp": {"status": "off", "brightness": 75}, "room_1_ac": {"status": "off", "temperature": 20}, "kitchen_ac": {"status": "off", "temperature": 20}, "living_room_tv": {"status": "off"}}


...

[FLASK_DEBUG] /process_text_command: Received command: 'turn off lamp room 1'
[AGENT_DEBUG] execute_command: Received original command: 'turn off lamp room 1'
[AGENT_DEBUG] execute_command: Detected lang: en
[AGENT_DEBUG] _get_llm_analysis: Sending to LLM. Command: "turn off lamp room 1"
[AGENT_DEBUG] _get_llm_analysis: LLM Raw Output: {
"intent": "control_device",
"device": "lamp",
"action": "turn_off",
"parameters": {"location": "room 1"}
}
[AGENT_DEBUG] _get_llm_analysis: LLM Parsed Output: {'intent': 'control_device', 'device': 'lamp', 'action': 'turn_off', 'parameters': {'location': 'room 1'}}
[AGENT_DEBUG] _is_valid_device_location: device='lamp', location='room 1', normalized='room 1', valid=True
[AGENT_DEBUG] _control_lamp: Action='turn_off', Location='room_1', Brightness='None', DeviceKey='room_1_lamp'
[AGENT_DEBUG] execute_command: Control device execution message: 'turned off.'
[AGENT_DEBUG] _generate_response: Intent='control_device', Lang='en', ExecMsg='turned off.', Final

INFO:werkzeug:127.0.0.1 - - [08/Jun/2025 15:05:34] "POST /process_text_command HTTP/1.1" 200 -


 > Processing time: 27.377105712890625
 > Real-time factor: 5.169782654230939
[FLASK_DEBUG] /process_text_command: TTS audio generated: /audio/a979e86c-886b-49fc-866d-2dd715aee279.wav
[FLASK_DEBUG] /process_text_command: Preparing to return JSON. text_response type: <class 'str'>, device_states type: <class 'dict'>
[FLASK_DEBUG] /process_text_command: Full response_payload: {'text_response': 'Okay, the lamp in room 1. Status: turned off.', 'audio_url': '/audio/a979e86c-886b-49fc-866d-2dd715aee279.wav', 'device_states': {'kitchen_lamp': {'status': 'off'}, 'bathroom_lamp': {'status': 'off', 'brightness': 75}, 'room_1_lamp': {'status': 'off'}, 'room_2_lamp': {'status': 'off', 'brightness': 75}, 'room_1_ac': {'status': 'off', 'temperature': 20}, 'kitchen_ac': {'status': 'off', 'temperature': 20}, 'living_room_tv': {'status': 'off'}}, 'error': None}


INFO:werkzeug:127.0.0.1 - - [08/Jun/2025 15:05:35] "GET /audio/a979e86c-886b-49fc-866d-2dd715aee279.wav?t=1749395135512 HTTP/1.1" 206 -


[FLASK_DEBUG] Serving audio file: a979e86c-886b-49fc-866d-2dd715aee279.wav


In [ ]:
ngrok.disconnect('https://6c07-34-91-135-82.ngrok-free.app')
ngrok.kill()

In [10]:
import requests

# Use the ngrok URL shown in your terminal
url = "  https://6650-143-14-203-35.ngrok-free.appp/ping"

# Send a ping to your PC
resp = requests.post(url, json={"source": "colab", "message": "Hello from Colab!"})



ConnectionError: HTTPSConnectionPool(host='6650-143-14-203-35.ngrok-free.appp', port=443): Max retries exceeded with url: /ping (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x788f19821e10>: Failed to resolve '6650-143-14-203-35.ngrok-free.appp' ([Errno -2] Name or service not known)"))

In [17]:
print(resp)

<Response [200]>


In [ ]:
import os
import signal

# Kill any process on port 5000
!lsof -t -i:5000 | xargs -r kill -9

# Optionally stop ngrok
try:
    ngrok.kill()
except:
    pass
